In [2]:
%pip install scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [1]:
# Verify the Python interpreter and installed package locations


import sys
import importlib.util
from pathlib import Path

print("Python executable:", sys.executable)
print("Python version:", sys.version)

for package in ["pandas", "numpy", "sklearn"]:
    spec = importlib.util.find_spec(package)
    print(f"{package} location:", spec.origin if spec else "Not found")

print("Working directory:", Path.cwd())

Python executable: D:\AtmoGraph-Supply-Chain-Ripple-Predictor\.venv\Scripts\python.exe
Python version: 3.11.5 (tags/v3.11.5:cce6ba9, Aug 24 2023, 14:38:34) [MSC v.1936 64 bit (AMD64)]
pandas location: D:\AtmoGraph-Supply-Chain-Ripple-Predictor\.venv\Lib\site-packages\pandas\__init__.py
numpy location: D:\AtmoGraph-Supply-Chain-Ripple-Predictor\.venv\Lib\site-packages\numpy\__init__.py
sklearn location: D:\AtmoGraph-Supply-Chain-Ripple-Predictor\.venv\Lib\site-packages\sklearn\__init__.py
Working directory: D:\AtmoGraph-Supply-Chain-Ripple-Predictor\notebooks


In [2]:

import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.preprocessing import StandardScaler, OneHotEncoder

print("Libraries imported successfully!")


Libraries imported successfully!


In [3]:

# Project directories

PROJECT_DIR = Path(
    r"D:\AtmoGraph-Supply-Chain-Ripple-Predictor"
)

DATA_DIR = Path(
    r"D:\AtmoGraph-Supply-Chain-Ripple-Predictor-Krish\graph_database\data"
)

OUTPUT_DIR = PROJECT_DIR / "outputs" / "processed_features"

NODE_OUTPUT_DIR = OUTPUT_DIR / "nodes"
EDGE_OUTPUT_DIR = OUTPUT_DIR / "edges"
METADATA_DIR = OUTPUT_DIR / "metadata"

# Create output directories
for folder in [NODE_OUTPUT_DIR, EDGE_OUTPUT_DIR, METADATA_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("Project paths configured successfully!")
print("Data directory exists:", DATA_DIR.exists())
print("Output directory:", OUTPUT_DIR)


Project paths configured successfully!
Data directory exists: True
Output directory: D:\AtmoGraph-Supply-Chain-Ripple-Predictor\outputs\processed_features


In [4]:

# Load node datasets

nodes = {
    "suppliers": pd.read_csv(DATA_DIR / "nodes_suppliers.csv"),
    "manufacturers": pd.read_csv(DATA_DIR / "nodes_manufacturers.csv"),
    "ports": pd.read_csv(DATA_DIR / "nodes_ports.csv"),
    "distributors": pd.read_csv(DATA_DIR / "nodes_distributors.csv"),
    "disruption_events": pd.read_csv(DATA_DIR / "nodes_disruption_events.csv")
}

# Load relationship datasets

edges = {
    "supplies_to": pd.read_csv(DATA_DIR / "edges_supplies_to.csv"),
    "ships_via": pd.read_csv(DATA_DIR / "edges_ships_via.csv"),
    "route_to": pd.read_csv(DATA_DIR / "edges_route_to.csv"),
    "delivers_to": pd.read_csv(DATA_DIR / "edges_delivers_to.csv"),
    "had_disruption": pd.read_csv(DATA_DIR / "edges_had_disruption.csv")
}

print("Updated graph datasets loaded successfully!")


Updated graph datasets loaded successfully!


In [5]:

# Cell 4: Verify Loaded Datasets

print("NODE DATASETS")
print("=" * 50)

total_nodes = 0

for name, df in nodes.items():
    print(f"{name}: {df.shape[0]} rows, {df.shape[1]} columns")
    total_nodes += len(df)

print("\nTotal nodes:", total_nodes)

print("\nRELATIONSHIP DATASETS")
print("=" * 50)

total_edges = 0

for name, df in edges.items():
    print(f"{name}: {df.shape[0]} rows, {df.shape[1]} columns")
    total_edges += len(df)

print("\nTotal relationships:", total_edges)

assert total_nodes == 45, "Unexpected node count!"
assert total_edges == 108, "Unexpected relationship count!"

print("\nDataset verification successful!")


NODE DATASETS
suppliers: 5 rows, 13 columns
manufacturers: 5 rows, 13 columns
ports: 13 rows, 12 columns
distributors: 10 rows, 8 columns
disruption_events: 12 rows, 10 columns

Total nodes: 45

RELATIONSHIP DATASETS
supplies_to: 25 rows, 8 columns
ships_via: 9 rows, 9 columns
route_to: 50 rows, 15 columns
delivers_to: 12 rows, 7 columns
had_disruption: 12 rows, 3 columns

Total relationships: 108

Dataset verification successful!


In [6]:

# Cell 5: Define Candidate Node Features

node_feature_candidates = {
    "suppliers": {
        "numerical": [
            "lat",
            "lon",
            "sku_count",
            "avg_defect_rate",
            "avg_supplier_lead_time_days",
            "inspection_fail_share"
        ],
        "categorical": [
            "primary_city"
        ]
    },

    "manufacturers": {
        "numerical": [
            "lat",
            "lon",
            "sku_count",
            "production_volume",
            "avg_manufacturing_lead_time_days",
            "avg_manufacturing_cost",
            "avg_defect_rate"
        ],
        "categorical": [
            "city"
        ]
    },

    "ports": {
        "numerical": [
            "lat",
            "lon",
            "hist_disruption_events",
            "hist_avg_affected_days",
            "hist_max_severity"
        ],
        "categorical": [
            "country"
        ]
    },

    "distributors": {
        "numerical": [
            "lat",
            "lon"
        ],
        "categorical": [
            "country"
        ]
    },

    "disruption_events": {
        "numerical": [
            "year"
        ],
        "categorical": []
    }
}

print("Candidate node features defined successfully!")


Candidate node features defined successfully!


In [7]:

# Cell 6: Validate Candidate Node Features

for node_type, feature_groups in node_feature_candidates.items():

    df = nodes[node_type]

    selected_columns = (
        feature_groups["numerical"]
        + feature_groups["categorical"]
    )

    print(f"\n{'=' * 55}")
    print(f"NODE TYPE: {node_type.upper()}")
    print("=" * 55)

    missing_columns = [
        col for col in selected_columns
        if col not in df.columns
    ]

    if missing_columns:
        print("Missing columns:", missing_columns)
    else:
        print("All selected columns are available.")

    print("\nMissing values:")
    display(df[selected_columns].isnull().sum())

    print("\nSelected feature count:", len(selected_columns))



NODE TYPE: SUPPLIERS
All selected columns are available.

Missing values:


lat                            0
lon                            0
sku_count                      0
avg_defect_rate                0
avg_supplier_lead_time_days    0
inspection_fail_share          0
primary_city                   0
dtype: int64


Selected feature count: 7

NODE TYPE: MANUFACTURERS
All selected columns are available.

Missing values:


lat                                 0
lon                                 0
sku_count                           0
production_volume                   0
avg_manufacturing_lead_time_days    0
avg_manufacturing_cost              0
avg_defect_rate                     0
city                                0
dtype: int64


Selected feature count: 8

NODE TYPE: PORTS
All selected columns are available.

Missing values:


lat                       0
lon                       0
hist_disruption_events    0
hist_avg_affected_days    9
hist_max_severity         9
country                   0
dtype: int64


Selected feature count: 6

NODE TYPE: DISTRIBUTORS
All selected columns are available.

Missing values:


lat        0
lon        0
country    0
dtype: int64


Selected feature count: 3

NODE TYPE: DISRUPTION_EVENTS
All selected columns are available.

Missing values:


year    0
dtype: int64


Selected feature count: 1


In [8]:

edge_feature_candidates = {
    "supplies_to": {
        "numerical": [
            "lead_time_days",
            "volume",
            "dependency_weight",
            "n_skus"
        ],
        "categorical": ["transport_mode"]
    },

    "ships_via": {
        "numerical": [
            "lead_time_days",
            "volume",
            "dependency_weight",
            "distance_km"
        ],
        "categorical": []
    },

    "route_to": {
        "numerical": [
            "lead_time_days",
            "volume",
            "distance_km",
            "avg_freight_cost_usd",
            "avg_port_congestion",
            "dependency_weight"
        ],
        "categorical": [
            "transport_mode",
            "cargo_type"
        ]
    },

   "delivers_to": {
    "numerical": [
        "lead_time_days",
        "volume",
        "dependency_weight"
    ],
    "categorical": []
},

    "had_disruption": {
        "numerical": [],
        "categorical": []
    }
}

print("Revised relationship feature candidates defined successfully!")


Revised relationship feature candidates defined successfully!


In [9]:

for edge_type, feature_groups in edge_feature_candidates.items():
    df = edges[edge_type]

    selected_columns = (
        feature_groups["numerical"] +
        feature_groups["categorical"]
    )

    print(f"\n{'=' * 55}")
    print(f"RELATIONSHIP TYPE: {edge_type.upper()}")
    print("=" * 55)

    missing_columns = [
        col for col in selected_columns
        if col not in df.columns
    ]

    if missing_columns:
        print("Missing columns:", missing_columns)
    elif selected_columns:
        print("All selected columns are available.")
        print("\nMissing values:")
        display(df[selected_columns].isnull().sum())
    else:
        print("No additional relationship features selected.")

    print("\nSelected feature count:", len(selected_columns))



RELATIONSHIP TYPE: SUPPLIES_TO
All selected columns are available.

Missing values:


lead_time_days       0
volume               0
dependency_weight    0
n_skus               0
transport_mode       0
dtype: int64


Selected feature count: 5

RELATIONSHIP TYPE: SHIPS_VIA
All selected columns are available.

Missing values:


lead_time_days       0
volume               0
dependency_weight    0
distance_km          0
dtype: int64


Selected feature count: 4

RELATIONSHIP TYPE: ROUTE_TO
All selected columns are available.

Missing values:


lead_time_days          0
volume                  0
distance_km             0
avg_freight_cost_usd    0
avg_port_congestion     0
dependency_weight       0
transport_mode          0
cargo_type              0
dtype: int64


Selected feature count: 8

RELATIONSHIP TYPE: DELIVERS_TO
All selected columns are available.

Missing values:


lead_time_days       0
volume               0
dependency_weight    0
dtype: int64


Selected feature count: 3

RELATIONSHIP TYPE: HAD_DISRUPTION
No additional relationship features selected.

Selected feature count: 0


In [10]:

for edge_type, feature_groups in edge_feature_candidates.items():

    df = edges[edge_type]

    print(f"\n{'=' * 60}")
    print(f"RELATIONSHIP: {edge_type.upper()}")
    print("=" * 60)

    for col in feature_groups["numerical"]:
        print(f"\n{col}")
        print("Unique values:", df[col].nunique())
        print(df[col].describe())

    for col in feature_groups["categorical"]:
        print(f"\n{col}")
        print(df[col].value_counts())



RELATIONSHIP: SUPPLIES_TO

lead_time_days
Unique values: 24
count    25.000000
mean     17.632000
std       6.218341
min       3.000000
25%      15.250000
50%      17.500000
75%      22.330000
max      28.000000
Name: lead_time_days, dtype: float64

volume
Unique values: 25
count     25.000000
mean     196.880000
std       94.956709
min        9.000000
25%      140.000000
50%      194.000000
75%      270.000000
max      406.000000
Name: volume, dtype: float64

dependency_weight
Unique values: 25
count    25.000000
mean      0.200000
std       0.098201
min       0.012300
25%       0.117300
50%       0.199400
75%       0.248000
max       0.431100
Name: dependency_weight, dtype: float64

n_skus
Unique values: 7
count    25.000000
mean      4.000000
std       1.581139
min       1.000000
25%       3.000000
50%       4.000000
75%       5.000000
max       8.000000
Name: n_skus, dtype: float64

transport_mode
transport_mode
Air     8
Rail    8
Road    7
Sea     2
Name: count, dtype: int64

RE

In [11]:

print("CONSTANT FEATURE CHECK")
print("=" * 65)

for edge_type, feature_groups in edge_feature_candidates.items():

    df = edges[edge_type]

    print(f"\n{edge_type.upper()}")
    print("-" * 50)

    selected_columns = (
        feature_groups["numerical"] +
        feature_groups["categorical"]
    )

    if not selected_columns:
        print("No additional features.")
        continue

    for col in selected_columns:
        unique_count = df[col].nunique()

        status = "CONSTANT" if unique_count <= 1 else "VARIABLE"

        print(f"{col}: {unique_count} unique values ({status})")


CONSTANT FEATURE CHECK

SUPPLIES_TO
--------------------------------------------------
lead_time_days: 24 unique values (VARIABLE)
volume: 25 unique values (VARIABLE)
dependency_weight: 25 unique values (VARIABLE)
n_skus: 7 unique values (VARIABLE)
transport_mode: 4 unique values (VARIABLE)

SHIPS_VIA
--------------------------------------------------
lead_time_days: 5 unique values (VARIABLE)
volume: 9 unique values (VARIABLE)
dependency_weight: 7 unique values (VARIABLE)
distance_km: 9 unique values (VARIABLE)

ROUTE_TO
--------------------------------------------------
lead_time_days: 31 unique values (VARIABLE)
volume: 50 unique values (VARIABLE)
distance_km: 50 unique values (VARIABLE)
avg_freight_cost_usd: 50 unique values (VARIABLE)
avg_port_congestion: 50 unique values (VARIABLE)
dependency_weight: 48 unique values (VARIABLE)
transport_mode: 4 unique values (VARIABLE)
cargo_type: 5 unique values (VARIABLE)

DELIVERS_TO
--------------------------------------------------
lead_tim

In [12]:

node_feature_frames = {}

for node_type, feature_groups in node_feature_candidates.items():

    df = nodes[node_type].copy()

    numerical_cols = feature_groups["numerical"]
    categorical_cols = feature_groups["categorical"]

    selected_cols = numerical_cols + categorical_cols

    # Create a separate feature dataframe
    feature_df = df[selected_cols].copy()

    # Handle missing numerical values
    for col in numerical_cols:

        if feature_df[col].isnull().any():

            # Preserve missingness information
            feature_df[f"{col}_missing"] = (
                feature_df[col].isnull().astype(int)
            )

            # Replace missing values with the median
            median_value = feature_df[col].median()

            feature_df[col] = feature_df[col].fillna(
                median_value
            )

    node_feature_frames[node_type] = feature_df

    print(f"\n{node_type.upper()}")
    print("-" * 45)
    print("Feature shape:", feature_df.shape)
    print("Remaining missing values:", feature_df.isnull().sum().sum())

print("\nNode feature datasets created successfully!")



SUPPLIERS
---------------------------------------------
Feature shape: (5, 7)
Remaining missing values: 0

MANUFACTURERS
---------------------------------------------
Feature shape: (5, 8)
Remaining missing values: 0

PORTS
---------------------------------------------
Feature shape: (13, 8)
Remaining missing values: 0

DISTRIBUTORS
---------------------------------------------
Feature shape: (10, 3)
Remaining missing values: 0

DISRUPTION_EVENTS
---------------------------------------------
Feature shape: (12, 1)
Remaining missing values: 0

Node feature datasets created successfully!


In [13]:

from sklearn.preprocessing import OneHotEncoder

encoded_node_features = {}
node_encoders = {}

for node_type, feature_groups in node_feature_candidates.items():

    feature_df = node_feature_frames[node_type].copy()
    categorical_cols = feature_groups["categorical"]

    if categorical_cols:

        encoder = OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False
        )

        encoded_array = encoder.fit_transform(
            feature_df[categorical_cols]
        )

        encoded_columns = encoder.get_feature_names_out(
            categorical_cols
        )

        encoded_df = pd.DataFrame(
            encoded_array,
            columns=encoded_columns,
            index=feature_df.index
        )

        feature_df = feature_df.drop(
            columns=categorical_cols
        )

        feature_df = pd.concat(
            [feature_df, encoded_df],
            axis=1
        )

        node_encoders[node_type] = encoder

    encoded_node_features[node_type] = feature_df

    print(f"\n{node_type.upper()}")
    print("-" * 45)
    print("Encoded feature shape:", feature_df.shape)
    print("Remaining categorical columns:",
          list(feature_df.select_dtypes(include=["object"]).columns))

print("\nCategorical encoding completed successfully!")



SUPPLIERS
---------------------------------------------
Encoded feature shape: (5, 9)
Remaining categorical columns: []

MANUFACTURERS
---------------------------------------------
Encoded feature shape: (5, 12)
Remaining categorical columns: []

PORTS
---------------------------------------------
Encoded feature shape: (13, 17)
Remaining categorical columns: []

DISTRIBUTORS
---------------------------------------------
Encoded feature shape: (10, 12)
Remaining categorical columns: []

DISRUPTION_EVENTS
---------------------------------------------
Encoded feature shape: (12, 1)
Remaining categorical columns: []

Categorical encoding completed successfully!


In [14]:

from sklearn.preprocessing import StandardScaler

scaled_node_features = {}
node_scalers = {}

for node_type, feature_groups in node_feature_candidates.items():

    feature_df = encoded_node_features[node_type].copy()

    # Original numerical columns
    numerical_cols = feature_groups["numerical"].copy()

    # Include missingness indicators as binary numerical features
    missing_cols = [
        col for col in feature_df.columns
        if col.endswith("_missing")
    ]

    scaling_cols = [
        col for col in numerical_cols
        if col in feature_df.columns
    ]

    # Scale continuous numerical features, not binary indicators
    scaler = StandardScaler()

    if scaling_cols:
        feature_df[scaling_cols] = scaler.fit_transform(
            feature_df[scaling_cols]
        )

        node_scalers[node_type] = scaler

    scaled_node_features[node_type] = feature_df

    print(f"\n{node_type.upper()}")
    print("-" * 45)
    print("Feature shape:", feature_df.shape)
    print("Scaled columns:", scaling_cols)
    print("Missingness indicators preserved:", missing_cols)

print("\nNode numerical scaling completed successfully!")



SUPPLIERS
---------------------------------------------
Feature shape: (5, 9)
Scaled columns: ['lat', 'lon', 'sku_count', 'avg_defect_rate', 'avg_supplier_lead_time_days', 'inspection_fail_share']
Missingness indicators preserved: []

MANUFACTURERS
---------------------------------------------
Feature shape: (5, 12)
Scaled columns: ['lat', 'lon', 'sku_count', 'production_volume', 'avg_manufacturing_lead_time_days', 'avg_manufacturing_cost', 'avg_defect_rate']
Missingness indicators preserved: []

PORTS
---------------------------------------------
Feature shape: (13, 17)
Scaled columns: ['lat', 'lon', 'hist_disruption_events', 'hist_avg_affected_days', 'hist_max_severity']
Missingness indicators preserved: ['hist_avg_affected_days_missing', 'hist_max_severity_missing']

DISTRIBUTORS
---------------------------------------------
Feature shape: (10, 12)
Scaled columns: ['lat', 'lon']
Missingness indicators preserved: []

DISRUPTION_EVENTS
---------------------------------------------
Fe

In [15]:

print("PROCESSED NODE FEATURE VALIDATION")
print("=" * 65)

for node_type, feature_df in scaled_node_features.items():

    original_count = len(nodes[node_type])
    processed_count = len(feature_df)

    print(f"\n{node_type.upper()}")
    print("-" * 50)

    print("Original nodes:", original_count)
    print("Processed rows:", processed_count)
    print("Feature columns:", feature_df.shape[1])

    print(
        "All numerical:",
        all(pd.api.types.is_numeric_dtype(dtype)
            for dtype in feature_df.dtypes)
    )

    print("Missing values:", feature_df.isnull().sum().sum())

    print(
        "Infinite values:",
        np.isinf(feature_df.to_numpy(dtype=float)).sum()
    )

    assert original_count == processed_count
    assert feature_df.isnull().sum().sum() == 0
    assert np.isfinite(feature_df.to_numpy(dtype=float)).all()

print("\nAll node feature validation checks passed!")


PROCESSED NODE FEATURE VALIDATION

SUPPLIERS
--------------------------------------------------
Original nodes: 5
Processed rows: 5
Feature columns: 9
All numerical: True
Missing values: 0
Infinite values: 0

MANUFACTURERS
--------------------------------------------------
Original nodes: 5
Processed rows: 5
Feature columns: 12
All numerical: True
Missing values: 0
Infinite values: 0

PORTS
--------------------------------------------------
Original nodes: 13
Processed rows: 13
Feature columns: 17
All numerical: True
Missing values: 0
Infinite values: 0

DISTRIBUTORS
--------------------------------------------------
Original nodes: 10
Processed rows: 10
Feature columns: 12
All numerical: True
Missing values: 0
Infinite values: 0

DISRUPTION_EVENTS
--------------------------------------------------
Original nodes: 12
Processed rows: 12
Feature columns: 1
All numerical: True
Missing values: 0
Infinite values: 0

All node feature validation checks passed!


In [16]:
#Create relationship feature datasets

edge_feature_frames = {}
edge_index_frames = {}

for edge_type, feature_groups in edge_feature_candidates.items():

    df = edges[edge_type].copy()

    # Preserve graph connectivity separately
    edge_index_frames[edge_type] = df[["start_id", "end_id"]].copy()

    # Select only model features
    selected_cols = (
        feature_groups["numerical"] +
        feature_groups["categorical"]
    )

    feature_df = df[selected_cols].copy()

    edge_feature_frames[edge_type] = feature_df

    print(f"\n{edge_type.upper()}")
    print("-" * 45)
    print("Relationship count:", len(df))
    print("Feature shape:", feature_df.shape)
    print("Connectivity shape:", edge_index_frames[edge_type].shape)
    print("Missing values:", feature_df.isnull().sum().sum())

print("\nRelationship feature datasets created successfully!")



SUPPLIES_TO
---------------------------------------------
Relationship count: 25
Feature shape: (25, 5)
Connectivity shape: (25, 2)
Missing values: 0

SHIPS_VIA
---------------------------------------------
Relationship count: 9
Feature shape: (9, 4)
Connectivity shape: (9, 2)
Missing values: 0

ROUTE_TO
---------------------------------------------
Relationship count: 50
Feature shape: (50, 8)
Connectivity shape: (50, 2)
Missing values: 0

DELIVERS_TO
---------------------------------------------
Relationship count: 12
Feature shape: (12, 3)
Connectivity shape: (12, 2)
Missing values: 0

HAD_DISRUPTION
---------------------------------------------
Relationship count: 12
Feature shape: (12, 0)
Connectivity shape: (12, 2)
Missing values: 0.0

Relationship feature datasets created successfully!


In [17]:

# Encode categorical features for each relationship type using One-Hot Encoding.

from sklearn.preprocessing import OneHotEncoder

encoded_edge_features = {}
edge_encoders = {}

for edge_type, feature_groups in edge_feature_candidates.items():

    feature_df = edge_feature_frames[edge_type].copy()
    categorical_cols = feature_groups["categorical"]

    if categorical_cols:

        encoder = OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False
        )

        encoded_array = encoder.fit_transform(
            feature_df[categorical_cols]
        )

        encoded_columns = encoder.get_feature_names_out(
            categorical_cols
        )

        encoded_df = pd.DataFrame(
            encoded_array,
            columns=encoded_columns,
            index=feature_df.index
        )

        feature_df = feature_df.drop(
            columns=categorical_cols
        )

        feature_df = pd.concat(
            [feature_df, encoded_df],
            axis=1
        )

        edge_encoders[edge_type] = encoder

    encoded_edge_features[edge_type] = feature_df

    print(f"\n{edge_type.upper()}")
    print("-" * 45)
    print("Encoded feature shape:", feature_df.shape)
    print(
        "Remaining categorical columns:",
        list(feature_df.select_dtypes(include=["object"]).columns)
    )

print("\nRelationship categorical encoding completed successfully!")



SUPPLIES_TO
---------------------------------------------
Encoded feature shape: (25, 8)
Remaining categorical columns: []

SHIPS_VIA
---------------------------------------------
Encoded feature shape: (9, 4)
Remaining categorical columns: []

ROUTE_TO
---------------------------------------------
Encoded feature shape: (50, 15)
Remaining categorical columns: []

DELIVERS_TO
---------------------------------------------
Encoded feature shape: (12, 3)
Remaining categorical columns: []

HAD_DISRUPTION
---------------------------------------------
Encoded feature shape: (12, 0)
Remaining categorical columns: []

Relationship categorical encoding completed successfully!


In [18]:

# Standardize numerical features separately for each relationship type.

from sklearn.preprocessing import StandardScaler

scaled_edge_features = {}
edge_scalers = {}

for edge_type, feature_groups in edge_feature_candidates.items():

    feature_df = encoded_edge_features[edge_type].copy()

    numerical_cols = [
        col for col in feature_groups["numerical"]
        if col in feature_df.columns
    ]

    if numerical_cols:

        scaler = StandardScaler()

        feature_df[numerical_cols] = scaler.fit_transform(
            feature_df[numerical_cols]
        )

        edge_scalers[edge_type] = scaler

    scaled_edge_features[edge_type] = feature_df

    print(f"\n{edge_type.upper()}")
    print("-" * 45)
    print("Feature shape:", feature_df.shape)
    print("Scaled numerical columns:", numerical_cols)

print("\nRelationship numerical scaling completed successfully!")



SUPPLIES_TO
---------------------------------------------
Feature shape: (25, 8)
Scaled numerical columns: ['lead_time_days', 'volume', 'dependency_weight', 'n_skus']

SHIPS_VIA
---------------------------------------------
Feature shape: (9, 4)
Scaled numerical columns: ['lead_time_days', 'volume', 'dependency_weight', 'distance_km']

ROUTE_TO
---------------------------------------------
Feature shape: (50, 15)
Scaled numerical columns: ['lead_time_days', 'volume', 'distance_km', 'avg_freight_cost_usd', 'avg_port_congestion', 'dependency_weight']

DELIVERS_TO
---------------------------------------------
Feature shape: (12, 3)
Scaled numerical columns: ['lead_time_days', 'volume', 'dependency_weight']

HAD_DISRUPTION
---------------------------------------------
Feature shape: (12, 0)
Scaled numerical columns: []

Relationship numerical scaling completed successfully!


In [19]:

# Validate processed relationship features and graph connectivity.

print("PROCESSED RELATIONSHIP FEATURE VALIDATION")
print("=" * 65)

for edge_type, feature_df in scaled_edge_features.items():

    original_count = len(edges[edge_type])
    processed_count = len(feature_df)
    connectivity_count = len(edge_index_frames[edge_type])

    print(f"\n{edge_type.upper()}")
    print("-" * 50)

    print("Original relationships:", original_count)
    print("Processed rows:", processed_count)
    print("Feature columns:", feature_df.shape[1])
    print("Connectivity rows:", connectivity_count)

    print(
        "All numerical:",
        all(
            pd.api.types.is_numeric_dtype(dtype)
            for dtype in feature_df.dtypes
        )
    )

    print("Missing values:", feature_df.isnull().sum().sum())

    infinite_count = (
        np.isinf(feature_df.to_numpy(dtype=float)).sum()
        if feature_df.shape[1] > 0 else 0
    )

    print("Infinite values:", infinite_count)

    assert original_count == processed_count
    assert processed_count == connectivity_count
    assert feature_df.isnull().sum().sum() == 0
    assert infinite_count == 0
    assert all(
        pd.api.types.is_numeric_dtype(dtype)
        for dtype in feature_df.dtypes
    )

print("\nAll relationship feature validation checks passed!")


PROCESSED RELATIONSHIP FEATURE VALIDATION

SUPPLIES_TO
--------------------------------------------------
Original relationships: 25
Processed rows: 25
Feature columns: 8
Connectivity rows: 25
All numerical: True
Missing values: 0
Infinite values: 0

SHIPS_VIA
--------------------------------------------------
Original relationships: 9
Processed rows: 9
Feature columns: 4
Connectivity rows: 9
All numerical: True
Missing values: 0
Infinite values: 0

ROUTE_TO
--------------------------------------------------
Original relationships: 50
Processed rows: 50
Feature columns: 15
Connectivity rows: 50
All numerical: True
Missing values: 0
Infinite values: 0

DELIVERS_TO
--------------------------------------------------
Original relationships: 12
Processed rows: 12
Feature columns: 3
Connectivity rows: 12
All numerical: True
Missing values: 0
Infinite values: 0

HAD_DISRUPTION
--------------------------------------------------
Original relationships: 12
Processed rows: 12
Feature columns: 0
C

In [20]:

# Inspect node identifiers and relationship endpoints before graph construction.

print("NODE DATASET COLUMNS")
print("=" * 65)

for node_type, df in nodes.items():

    print(f"\n{node_type.upper()}")
    print("Columns:", df.columns.tolist())
    print("First 3 records:")
    display(df.head(3))

print("\nRELATIONSHIP ENDPOINTS")
print("=" * 65)

for edge_type, df in edge_index_frames.items():

    print(f"\n{edge_type.upper()}")
    print("First 5 connections:")
    display(df.head())


NODE DATASET COLUMNS

SUPPLIERS
Columns: ['id', 'name', 'country', 'country_iso3', 'lat', 'lon', 'primary_city', 'sku_count', 'avg_defect_rate', 'avg_supplier_lead_time_days', 'inspection_fail_share', 'geo_source', 'source']
First 3 records:


,id,name,country,country_iso3,lat,lon,primary_city,sku_count,avg_defect_rate,avg_supplier_lead_time_days,inspection_fail_share,geo_source,source
0,SUP_001,Supplier 1,India,IND,22.5726,88.3639,Kolkata,27,1.804,14.78,0.222,derived_modal_city,derived
1,SUP_002,Supplier 2,India,IND,28.6139,77.2090,Delhi,22,2.363,18.55,0.364,derived_modal_city,derived
2,SUP_003,Supplier 3,India,IND,13.0827,80.2707,Chennai,15,2.466,20.13,0.200,derived_modal_city,derived



MANUFACTURERS
Columns: ['id', 'name', 'country', 'country_iso3', 'lat', 'lon', 'city', 'sku_count', 'production_volume', 'avg_manufacturing_lead_time_days', 'avg_manufacturing_cost', 'avg_defect_rate', 'source']
First 3 records:


,id,name,country,country_iso3,lat,lon,city,sku_count,production_volume,avg_manufacturing_lead_time_days,avg_manufacturing_cost,avg_defect_rate,source
0,MFG_MUM,Mumbai Manufacturing Hub,India,IND,19.0760,72.8777,Mumbai,22,13160,19.73,36.73,2.122,derived
1,MFG_DEL,Delhi Manufacturing Hub,India,IND,28.6139,77.2090,Delhi,15,8362,13.53,48.88,2.229,derived
2,MFG_BLR,Bangalore Manufacturing Hub,India,IND,12.9716,77.5946,Bangalore,18,7827,11.78,61.99,2.094,derived



PORTS
Columns: ['id', 'name', 'unlocode', 'country', 'country_iso3', 'lat', 'lon', 'hist_disruption_events', 'hist_avg_affected_days', 'hist_max_severity', 'geo_source', 'source']
First 3 records:


,id,name,unlocode,country,country_iso3,lat,lon,hist_disruption_events,hist_avg_affected_days,hist_max_severity,geo_source,source
0,PORT_INNSA,Nhava Sheva (JNPT),INNSA,India,IND,18.95,72.95,0,NaN,NaN,approximate_manual,reference
1,PORT_INMUN,Mundra,INMUN,India,IND,22.74,69.70,0,NaN,NaN,approximate_manual,reference
2,PORT_INMAA,Chennai,INMAA,India,IND,13.09,80.29,0,NaN,NaN,approximate_manual,reference


... (10 outputs left out)


DELIVERS_TO
First 5 connections:


,start_id,end_id
0,PORT_CNSHA,DIST_CHN_01
1,PORT_USNYC,DIST_USA_01
2,PORT_JPTYO,DIST_JPN_01
3,PORT_DEHAM,DIST_DEU_01
4,PORT_GBFXT,DIST_GBR_01



HAD_DISRUPTION
First 5 connections:


,start_id,end_id
0,PORT_CNSHA,EVT_001
1,PORT_USNYC,EVT_002
2,PORT_USNYC,EVT_003
3,PORT_USNYC,EVT_004
4,PORT_USNYC,EVT_005


In [21]:

# Create a global numerical index mapping for all graph nodes.

node_id_to_index = {}
node_type_by_id = {}
node_index_maps = {}

current_index = 0

for node_type, df in nodes.items():

    node_index_maps[node_type] = {}

    for node_id in df["id"]:

        if node_id in node_id_to_index:
            raise ValueError(f"Duplicate node ID found: {node_id}")

        node_id_to_index[node_id] = current_index
        node_index_maps[node_type][node_id] = current_index
        node_type_by_id[node_id] = node_type

        current_index += 1

print("NODE ID MAPPING")
print("=" * 55)

print("Total mapped nodes:", len(node_id_to_index))

for node_type, mapping in node_index_maps.items():

    print(f"\n{node_type.upper()}")
    print("Mapped nodes:", len(mapping))
    print("Sample mapping:", list(mapping.items())[:3])

print("\nGlobal node ID mapping created successfully!")


NODE ID MAPPING
Total mapped nodes: 45

SUPPLIERS
Mapped nodes: 5
Sample mapping: [('SUP_001', 0), ('SUP_002', 1), ('SUP_003', 2)]

MANUFACTURERS
Mapped nodes: 5
Sample mapping: [('MFG_MUM', 5), ('MFG_DEL', 6), ('MFG_BLR', 7)]

PORTS
Mapped nodes: 13
Sample mapping: [('PORT_INNSA', 10), ('PORT_INMUN', 11), ('PORT_INMAA', 12)]

DISTRIBUTORS
Mapped nodes: 10
Sample mapping: [('DIST_CHN_01', 23), ('DIST_USA_01', 24), ('DIST_JPN_01', 25)]

DISRUPTION_EVENTS
Mapped nodes: 12
Sample mapping: [('EVT_001', 33), ('EVT_002', 34), ('EVT_003', 35)]

Global node ID mapping created successfully!


In [22]:

# Validate relationship endpoints against the global node ID mapping.

expected_edge_types = {
    "supplies_to": ("suppliers", "manufacturers"),
    "ships_via": ("manufacturers", "ports"),
    "route_to": ("ports", "ports"),
    "delivers_to": ("ports", "distributors"),
    "had_disruption": ("ports", "disruption_events")
}

print("GRAPH CONNECTIVITY VALIDATION")
print("=" * 65)

total_valid_edges = 0

for edge_type, df in edge_index_frames.items():

    source_type, destination_type = expected_edge_types[edge_type]

    source_ids = df["start_id"]
    destination_ids = df["end_id"]

    missing_sources = [
        node_id for node_id in source_ids
        if node_id not in node_id_to_index
    ]

    missing_destinations = [
        node_id for node_id in destination_ids
        if node_id not in node_id_to_index
    ]

    wrong_source_types = [
        node_id for node_id in source_ids
        if node_id in node_type_by_id
        and node_type_by_id[node_id] != source_type
    ]

    wrong_destination_types = [
        node_id for node_id in destination_ids
        if node_id in node_type_by_id
        and node_type_by_id[node_id] != destination_type
    ]

    valid = (
        not missing_sources
        and not missing_destinations
        and not wrong_source_types
        and not wrong_destination_types
    )

    print(f"\n{edge_type.upper()}")
    print("Relationship count:", len(df))
    print("Missing source IDs:", len(missing_sources))
    print("Missing destination IDs:", len(missing_destinations))
    print("Incorrect source types:", len(wrong_source_types))
    print("Incorrect destination types:", len(wrong_destination_types))
    print("Validation:", "PASSED" if valid else "FAILED")

    assert valid, f"Connectivity validation failed for {edge_type}"

    total_valid_edges += len(df)

print("\nTotal valid relationships:", total_valid_edges)
assert total_valid_edges == 108

print("All graph connectivity checks passed!")


GRAPH CONNECTIVITY VALIDATION

SUPPLIES_TO
Relationship count: 25
Missing source IDs: 0
Missing destination IDs: 0
Incorrect source types: 0
Incorrect destination types: 0
Validation: PASSED

SHIPS_VIA
Relationship count: 9
Missing source IDs: 0
Missing destination IDs: 0
Incorrect source types: 0
Incorrect destination types: 0
Validation: PASSED

ROUTE_TO
Relationship count: 50
Missing source IDs: 0
Missing destination IDs: 0
Incorrect source types: 0
Incorrect destination types: 0
Validation: PASSED

DELIVERS_TO
Relationship count: 12
Missing source IDs: 0
Missing destination IDs: 0
Incorrect source types: 0
Incorrect destination types: 0
Validation: PASSED

HAD_DISRUPTION
Relationship count: 12
Missing source IDs: 0
Missing destination IDs: 0
Incorrect source types: 0
Incorrect destination types: 0
Validation: PASSED

Total valid relationships: 108
All graph connectivity checks passed!


In [23]:

# Validate relationship endpoints against the global node ID mapping.

expected_edge_types = {
    "supplies_to": ("suppliers", "manufacturers"),
    "ships_via": ("manufacturers", "ports"),
    "route_to": ("ports", "ports"),
    "delivers_to": ("ports", "distributors"),
    "had_disruption": ("ports", "disruption_events")
}

print("GRAPH CONNECTIVITY VALIDATION")
print("=" * 65)

total_valid_edges = 0

for edge_type, df in edge_index_frames.items():

    source_type, destination_type = expected_edge_types[edge_type]

    source_ids = df["start_id"]
    destination_ids = df["end_id"]

    missing_sources = [
        node_id for node_id in source_ids
        if node_id not in node_id_to_index
    ]

    missing_destinations = [
        node_id for node_id in destination_ids
        if node_id not in node_id_to_index
    ]

    wrong_source_types = [
        node_id for node_id in source_ids
        if node_id in node_type_by_id
        and node_type_by_id[node_id] != source_type
    ]

    wrong_destination_types = [
        node_id for node_id in destination_ids
        if node_id in node_type_by_id
        and node_type_by_id[node_id] != destination_type
    ]

    valid = (
        not missing_sources
        and not missing_destinations
        and not wrong_source_types
        and not wrong_destination_types
    )

    print(f"\n{edge_type.upper()}")
    print("Relationship count:", len(df))
    print("Missing source IDs:", len(missing_sources))
    print("Missing destination IDs:", len(missing_destinations))
    print("Incorrect source types:", len(wrong_source_types))
    print("Incorrect destination types:", len(wrong_destination_types))
    print("Validation:", "PASSED" if valid else "FAILED")

    assert valid, f"Connectivity validation failed for {edge_type}"

    total_valid_edges += len(df)

print("\nTotal valid relationships:", total_valid_edges)
assert total_valid_edges == 108

print("All graph connectivity checks passed!")


GRAPH CONNECTIVITY VALIDATION

SUPPLIES_TO
Relationship count: 25
Missing source IDs: 0
Missing destination IDs: 0
Incorrect source types: 0
Incorrect destination types: 0
Validation: PASSED

SHIPS_VIA
Relationship count: 9
Missing source IDs: 0
Missing destination IDs: 0
Incorrect source types: 0
Incorrect destination types: 0
Validation: PASSED

ROUTE_TO
Relationship count: 50
Missing source IDs: 0
Missing destination IDs: 0
Incorrect source types: 0
Incorrect destination types: 0
Validation: PASSED

DELIVERS_TO
Relationship count: 12
Missing source IDs: 0
Missing destination IDs: 0
Incorrect source types: 0
Incorrect destination types: 0
Validation: PASSED

HAD_DISRUPTION
Relationship count: 12
Missing source IDs: 0
Missing destination IDs: 0
Incorrect source types: 0
Incorrect destination types: 0
Validation: PASSED

Total valid relationships: 108
All graph connectivity checks passed!


In [24]:

# Convert graph relationships into numerical edge index arrays.

edge_index_arrays = {}

for edge_type, df in edge_index_frames.items():

    source_indices = df["start_id"].map(node_id_to_index).to_numpy()
    destination_indices = df["end_id"].map(node_id_to_index).to_numpy()

    edge_index = np.vstack([
        source_indices,
        destination_indices
    ])

    edge_index_arrays[edge_type] = edge_index

    print(f"{edge_type.upper()}")
    print("Edge index shape:", edge_index.shape)
    print("First 3 edges:")
    print(edge_index[:, :3])
    print("-" * 50)

print("All relationship types converted into numerical edge indices!")


SUPPLIES_TO
Edge index shape: (2, 25)
First 3 edges:
[[0 0 0]
 [7 8 6]]
--------------------------------------------------
SHIPS_VIA
Edge index shape: (2, 9)
First 3 edges:
[[ 5  5  6]
 [10 11 11]]
--------------------------------------------------
ROUTE_TO
Edge index shape: (2, 50)
First 3 edges:
[[10 16 22]
 [15 14 14]]
--------------------------------------------------
DELIVERS_TO
Edge index shape: (2, 12)
First 3 edges:
[[14 15 16]
 [23 24 25]]
--------------------------------------------------
HAD_DISRUPTION
Edge index shape: (2, 12)
First 3 edges:
[[14 15 15]
 [33 34 35]]
--------------------------------------------------
All relationship types converted into numerical edge indices!


In [25]:

# Convert processed node and relationship features into numerical matrices.

node_feature_arrays = {}
edge_feature_arrays = {}

print("NODE FEATURE MATRICES")
print("=" * 55)

for node_type, df in scaled_node_features.items():

    feature_array = df.to_numpy(dtype=np.float32)

    node_feature_arrays[node_type] = feature_array

    print(f"{node_type.upper()}:")
    print("Feature matrix shape:", feature_array.shape)
    print("Data type:", feature_array.dtype)

print("\nEDGE FEATURE MATRICES")
print("=" * 55)

for edge_type, df in scaled_edge_features.items():

    feature_array = df.to_numpy(dtype=np.float32)

    edge_feature_arrays[edge_type] = feature_array

    print(f"{edge_type.upper()}:")
    print("Feature matrix shape:", feature_array.shape)
    print("Data type:", feature_array.dtype)

print("\nFeature matrices created successfully!")


NODE FEATURE MATRICES
SUPPLIERS:
Feature matrix shape: (5, 9)
Data type: float32
MANUFACTURERS:
Feature matrix shape: (5, 12)
Data type: float32
PORTS:
Feature matrix shape: (13, 17)
Data type: float32
DISTRIBUTORS:
Feature matrix shape: (10, 12)
Data type: float32
DISRUPTION_EVENTS:
Feature matrix shape: (12, 1)
Data type: float32

EDGE FEATURE MATRICES
SUPPLIES_TO:
Feature matrix shape: (25, 8)
Data type: float32
SHIPS_VIA:
Feature matrix shape: (9, 4)
Data type: float32
ROUTE_TO:
Feature matrix shape: (50, 15)
Data type: float32
DELIVERS_TO:
Feature matrix shape: (12, 3)
Data type: float32
HAD_DISRUPTION:
Feature matrix shape: (12, 0)
Data type: float32

Feature matrices created successfully!


In [26]:
# Inspect the structure of processed node features, edge features, and graph mappings

objects_to_check = [
    "scaled_node_features",
    "scaled_edge_features",
    "edge_index_frames",
    "node_id_to_index",
    "node_index_maps",
    "node_type_by_id"
]

for name in objects_to_check:
    obj = globals().get(name)

    print(f"\n{name}:")
    print("Type:", type(obj).__name__)

    if isinstance(obj, dict):
        print("Keys:", list(obj.keys()))

        for key, value in obj.items():
            print(
                f"  {key}: Type={type(value).__name__}, "
                f"Shape={getattr(value, 'shape', 'N/A')}"
            )
    else:
        print("Shape:", getattr(obj, "shape", "N/A"))
        print("Length:", len(obj) if obj is not None else "N/A")


scaled_node_features:
Type: dict
Keys: ['suppliers', 'manufacturers', 'ports', 'distributors', 'disruption_events']
  suppliers: Type=DataFrame, Shape=(5, 9)
  manufacturers: Type=DataFrame, Shape=(5, 12)
  ports: Type=DataFrame, Shape=(13, 17)
  distributors: Type=DataFrame, Shape=(10, 12)
  disruption_events: Type=DataFrame, Shape=(12, 1)

scaled_edge_features:
Type: dict
Keys: ['supplies_to', 'ships_via', 'route_to', 'delivers_to', 'had_disruption']
  supplies_to: Type=DataFrame, Shape=(25, 8)
  ships_via: Type=DataFrame, Shape=(9, 4)
  route_to: Type=DataFrame, Shape=(50, 15)
  delivers_to: Type=DataFrame, Shape=(12, 3)
  had_disruption: Type=DataFrame, Shape=(12, 0)

edge_index_frames:
Type: dict
Keys: ['supplies_to', 'ships_via', 'route_to', 'delivers_to', 'had_disruption']
  supplies_to: Type=DataFrame, Shape=(25, 2)
  ships_via: Type=DataFrame, Shape=(9, 2)
  route_to: Type=DataFrame, Shape=(50, 2)
  delivers_to: Type=DataFrame, Shape=(12, 2)
  had_disruption: Type=DataFrame, 

In [27]:
# Inspect relationship endpoint columns before exporting GNN edge data

for relation, df in edge_index_frames.items():
    print(f"\n{relation.upper()}")
    print("Columns:", df.columns.tolist())
    print(df.head(3))


SUPPLIES_TO
Columns: ['start_id', 'end_id']
  start_id   end_id
0  SUP_001  MFG_BLR
1  SUP_001  MFG_MAA
2  SUP_001  MFG_DEL

SHIPS_VIA
Columns: ['start_id', 'end_id']
  start_id      end_id
0  MFG_MUM  PORT_INNSA
1  MFG_MUM  PORT_INMUN
2  MFG_DEL  PORT_INMUN

ROUTE_TO
Columns: ['start_id', 'end_id']
     start_id      end_id
0  PORT_INNSA  PORT_USNYC
1  PORT_JPTYO  PORT_CNSHA
2  PORT_AUBNE  PORT_CNSHA

DELIVERS_TO
Columns: ['start_id', 'end_id']
     start_id       end_id
0  PORT_CNSHA  DIST_CHN_01
1  PORT_USNYC  DIST_USA_01
2  PORT_JPTYO  DIST_JPN_01

HAD_DISRUPTION
Columns: ['start_id', 'end_id']
     start_id   end_id
0  PORT_CNSHA  EVT_001
1  PORT_USNYC  EVT_002
2  PORT_USNYC  EVT_003


In [28]:
# Export processed GNN node features, edge features, and graph metadata

import json

# Export node feature matrices
for node_type, feature_df in scaled_node_features.items():
    original_nodes = nodes[node_type]

    if not feature_df.index.equals(original_nodes.index):
        raise ValueError(f"Node row alignment failed: {node_type}")

    export_df = feature_df.reset_index(drop=True).copy()
    node_ids = original_nodes["id"].reset_index(drop=True)

    export_df.insert(0, "node_id", node_ids)
    export_df.insert(
        1,
        "node_index",
        node_ids.map(node_id_to_index)
    )

    if export_df["node_index"].isnull().any():
        raise ValueError(f"Missing node index: {node_type}")

    output_path = NODE_OUTPUT_DIR / f"{node_type}_features.csv"
    export_df.to_csv(output_path, index=False)

    print(f"Saved {node_type}: {export_df.shape}")


# Export edge feature matrices with graph connectivity
for relation, feature_df in scaled_edge_features.items():
    endpoints = edge_index_frames[relation]

    if not feature_df.index.equals(endpoints.index):
        raise ValueError(f"Edge row alignment failed: {relation}")

    export_df = endpoints.reset_index(drop=True).copy()
    feature_values = feature_df.reset_index(drop=True)

    export_df["source_index"] = export_df["start_id"].map(node_id_to_index)
    export_df["target_index"] = export_df["end_id"].map(node_id_to_index)

    if export_df[["source_index", "target_index"]].isnull().any().any():
        raise ValueError(f"Missing endpoint mapping: {relation}")

    export_df = pd.concat([export_df, feature_values], axis=1)

    output_path = EDGE_OUTPUT_DIR / f"{relation}_features.csv"
    export_df.to_csv(output_path, index=False)

    print(f"Saved {relation}: {export_df.shape}")


# Save graph mappings and feature metadata
metadata = {
    "node_id_to_index": node_id_to_index,
    "node_type_by_id": node_type_by_id,
    "node_index_maps": node_index_maps,
    "node_feature_columns": {
        key: value.columns.tolist()
        for key, value in scaled_node_features.items()
    },
    "edge_feature_columns": {
        key: value.columns.tolist()
        for key, value in scaled_edge_features.items()
    }
}

with open(METADATA_DIR / "gnn_feature_metadata.json", "w") as file:
    json.dump(metadata, file, indent=4)

print("\nGNN feature export completed successfully!")
print("Output directory:", OUTPUT_DIR)

Saved suppliers: (5, 11)
Saved manufacturers: (5, 14)
Saved ports: (13, 19)
Saved distributors: (10, 14)
Saved disruption_events: (12, 3)
Saved supplies_to: (25, 12)
Saved ships_via: (9, 8)
Saved route_to: (50, 19)
Saved delivers_to: (12, 7)
Saved had_disruption: (12, 4)

GNN feature export completed successfully!
Output directory: D:\AtmoGraph-Supply-Chain-Ripple-Predictor\outputs\processed_features


In [29]:
# Validate exported GNN feature files and graph connectivity

print("NODE FEATURE VALIDATION")

for node_type in scaled_node_features:
    file_path = NODE_OUTPUT_DIR / f"{node_type}_features.csv"
    df = pd.read_csv(file_path)

    print(
        f"{node_type}: Rows={len(df)}, "
        f"Columns={len(df.columns)}, "
        f"Missing={df.isnull().sum().sum()}"
    )

print("\nEDGE FEATURE VALIDATION")

for relation in scaled_edge_features:
    file_path = EDGE_OUTPUT_DIR / f"{relation}_features.csv"
    df = pd.read_csv(file_path)

    valid_sources = df["source_index"].notna().all()
    valid_targets = df["target_index"].notna().all()

    print(
        f"{relation}: Rows={len(df)}, "
        f"Columns={len(df.columns)}, "
        f"Source mapping={valid_sources}, "
        f"Target mapping={valid_targets}"
    )

metadata_path = METADATA_DIR / "gnn_feature_metadata.json"

with open(metadata_path, "r") as file:
    saved_metadata = json.load(file)

print("\nMETADATA VALIDATION")
print("Node mappings:", len(saved_metadata["node_id_to_index"]))
print("Node types:", len(saved_metadata["node_index_maps"]))
print("Metadata file exists:", metadata_path.exists())

print("\nExport validation completed!")

NODE FEATURE VALIDATION
suppliers: Rows=5, Columns=11, Missing=0
manufacturers: Rows=5, Columns=14, Missing=0
ports: Rows=13, Columns=19, Missing=0
distributors: Rows=10, Columns=14, Missing=0
disruption_events: Rows=12, Columns=3, Missing=0

EDGE FEATURE VALIDATION
supplies_to: Rows=25, Columns=12, Source mapping=True, Target mapping=True
ships_via: Rows=9, Columns=8, Source mapping=True, Target mapping=True
route_to: Rows=50, Columns=19, Source mapping=True, Target mapping=True
delivers_to: Rows=12, Columns=7, Source mapping=True, Target mapping=True
had_disruption: Rows=12, Columns=4, Source mapping=True, Target mapping=True

METADATA VALIDATION
Node mappings: 45
Node types: 5
Metadata file exists: True

Export validation completed!


In [30]:
# Check PyTorch Geometric availability and local node index mappings

import importlib.util

print("PyTorch Geometric installed:",
      importlib.util.find_spec("torch_geometric") is not None)

print("\nLOCAL NODE INDEX MAPPINGS")

for node_type, mapping in node_index_maps.items():
    print(f"\n{node_type}:")
    print(mapping)

print("\nMETADATA FILE:")
print("Exists:", (METADATA_DIR / "gnn_feature_metadata.json").exists())
print("Saved node mappings:", len(saved_metadata["node_id_to_index"]))

PyTorch Geometric installed: False

LOCAL NODE INDEX MAPPINGS

suppliers:
{'SUP_001': 0, 'SUP_002': 1, 'SUP_003': 2, 'SUP_004': 3, 'SUP_005': 4}

manufacturers:
{'MFG_MUM': 5, 'MFG_DEL': 6, 'MFG_BLR': 7, 'MFG_MAA': 8, 'MFG_CCU': 9}

ports:
{'PORT_INNSA': 10, 'PORT_INMUN': 11, 'PORT_INMAA': 12, 'PORT_INCCU': 13, 'PORT_CNSHA': 14, 'PORT_USNYC': 15, 'PORT_JPTYO': 16, 'PORT_DEHAM': 17, 'PORT_GBFXT': 18, 'PORT_FRLEH': 19, 'PORT_BRSSZ': 20, 'PORT_CAVAN': 21, 'PORT_AUBNE': 22}

distributors:
{'DIST_CHN_01': 23, 'DIST_USA_01': 24, 'DIST_JPN_01': 25, 'DIST_DEU_01': 26, 'DIST_GBR_01': 27, 'DIST_FRA_01': 28, 'DIST_BRA_01': 29, 'DIST_CAN_01': 30, 'DIST_AUS_01': 31, 'DIST_IND_01': 32}

disruption_events:
{'EVT_001': 33, 'EVT_002': 34, 'EVT_003': 35, 'EVT_004': 36, 'EVT_005': 37, 'EVT_006': 38, 'EVT_007': 39, 'EVT_008': 40, 'EVT_009': 41, 'EVT_010': 42, 'EVT_011': 43, 'EVT_012': 44}

METADATA FILE:
Exists: True
Saved node mappings: 45


In [31]:
# Check PyTorch installation and version before setting up PyTorch Geometric

import importlib.util

torch_available = importlib.util.find_spec("torch") is not None

print("PyTorch installed:", torch_available)

if torch_available:
    import torch
    print("PyTorch version:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
else:
    print("PyTorch needs to be installed before PyTorch Geometric.")

PyTorch installed: False
PyTorch needs to be installed before PyTorch Geometric.


In [32]:
# Verify PyTorch and PyTorch Geometric inside the AtmoGraph notebook kernel

import torch
import torch_geometric
from torch_geometric.data import HeteroData

print("PyTorch:", torch.__version__)
print("PyTorch Geometric:", torch_geometric.__version__)
print("Heterogeneous graph support: Available")
print("CPU available:", torch.device("cpu"))

PyTorch: 2.15.0+cpu
PyTorch Geometric: 2.8.1
Heterogeneous graph support: Available
CPU available: cpu


D:\AtmoGraph-Supply-Chain-Ripple-Predictor\.venv\Lib\site-packages\torch\jit\_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


In [33]:
# Construct the AtmoGraph heterogeneous graph using PyTorch Geometric

import torch
from torch_geometric.data import HeteroData

# Define node types and relationship directions
node_types = [
    "suppliers",
    "manufacturers",
    "ports",
    "distributors",
    "disruption_events"
]

edge_type_mapping = {
    "supplies_to": ("suppliers", "supplies_to", "manufacturers"),
    "ships_via": ("manufacturers", "ships_via", "ports"),
    "route_to": ("ports", "route_to", "ports"),
    "delivers_to": ("ports", "delivers_to", "distributors"),
    "had_disruption": ("ports", "had_disruption", "disruption_events")
}

# Create separate local node indices for each node type
node_local_id_maps = {}

for node_type in node_types:
    node_local_id_maps[node_type] = {
        node_id: local_index
        for local_index, node_id in enumerate(node_index_maps[node_type].keys())
    }

# Initialize heterogeneous graph
hetero_data = HeteroData()

# Add node feature tensors
for node_type in node_types:
    feature_df = scaled_node_features[node_type]
    original_ids = nodes[node_type]["id"].tolist()

    if original_ids != list(node_local_id_maps[node_type].keys()):
        raise ValueError(f"Node ordering mismatch: {node_type}")

    node_features = torch.tensor(
        feature_df.to_numpy(dtype=np.float32),
        dtype=torch.float32
    )

    hetero_data[node_type].x = node_features

    print(
        f"Added {node_type}: "
        f"{node_features.shape[0]} nodes, "
        f"{node_features.shape[1]} features"
    )

# Add relationship connectivity and edge features
for relation, edge_type in edge_type_mapping.items():
    source_type, _, target_type = edge_type

    endpoints = edge_index_frames[relation]
    feature_df = scaled_edge_features[relation]

    source_indices = endpoints["start_id"].map(
        node_local_id_maps[source_type]
    )
    target_indices = endpoints["end_id"].map(
        node_local_id_maps[target_type]
    )

    if source_indices.isnull().any() or target_indices.isnull().any():
        raise ValueError(f"Invalid node mapping in {relation}")

    edge_index = torch.tensor(
        [source_indices.to_numpy(), target_indices.to_numpy()],
        dtype=torch.long
    )

    hetero_data[edge_type].edge_index = edge_index

    if feature_df.shape[1] > 0:
        edge_features = torch.tensor(
            feature_df.to_numpy(dtype=np.float32),
            dtype=torch.float32
        )
        hetero_data[edge_type].edge_attr = edge_features

    print(f"Added {relation}: {edge_index.shape[1]} relationships")

# Validate heterogeneous graph
hetero_data.validate(raise_on_error=True)

print("\nAtmoGraph heterogeneous graph created successfully!")
print(hetero_data)

Added suppliers: 5 nodes, 9 features
Added manufacturers: 5 nodes, 12 features
Added ports: 13 nodes, 17 features
Added distributors: 10 nodes, 12 features
Added disruption_events: 12 nodes, 1 features
Added supplies_to: 25 relationships
Added ships_via: 9 relationships
Added route_to: 50 relationships
Added delivers_to: 12 relationships
Added had_disruption: 12 relationships

AtmoGraph heterogeneous graph created successfully!
HeteroData(
  suppliers={ x=[5, 9] },
  manufacturers={ x=[5, 12] },
  ports={ x=[13, 17] },
  distributors={ x=[10, 12] },
  disruption_events={ x=[12, 1] },
  (suppliers, supplies_to, manufacturers)={
    edge_index=[2, 25],
    edge_attr=[25, 8],
  },
  (manufacturers, ships_via, ports)={
    edge_index=[2, 9],
    edge_attr=[9, 4],
  },
  (ports, route_to, ports)={
    edge_index=[2, 50],
    edge_attr=[50, 15],
  },
  (ports, delivers_to, distributors)={
    edge_index=[2, 12],
    edge_attr=[12, 3],
  },
  (ports, had_disruption, disruption_events)={ edge_

C:\Users\pedda\AppData\Local\Temp\ipykernel_17176\2592279525.py:73: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\torch\csrc\utils\tensor_new.cpp:252.)
  edge_index = torch.tensor(


In [34]:
# Validate all node and edge tensors in the AtmoGraph heterogeneous graph

print("NODE TENSOR VALIDATION")

for node_type in hetero_data.node_types:
    x = hetero_data[node_type].x

    print(
        f"{node_type}: Shape={tuple(x.shape)}, "
        f"Data type={x.dtype}, "
        f"Valid={torch.isfinite(x).all().item()}"
    )

print("\nEDGE TENSOR VALIDATION")

for edge_type in hetero_data.edge_types:
    edge_index = hetero_data[edge_type].edge_index

    source_type, _, target_type = edge_type

    source_count = hetero_data[source_type].num_nodes
    target_count = hetero_data[target_type].num_nodes

    valid_indices = (
        edge_index.min().item() >= 0
        and edge_index[0].max().item() < source_count
        and edge_index[1].max().item() < target_count
    )

    print(
        f"{edge_type}: "
        f"Shape={tuple(edge_index.shape)}, "
        f"Valid indices={valid_indices}"
    )

    if "edge_attr" in hetero_data[edge_type]:
        print(
            "  Edge features:",
            tuple(hetero_data[edge_type].edge_attr.shape)
        )

print("\nGraph tensor validation completed!")

NODE TENSOR VALIDATION
suppliers: Shape=(5, 9), Data type=torch.float32, Valid=True
manufacturers: Shape=(5, 12), Data type=torch.float32, Valid=True
ports: Shape=(13, 17), Data type=torch.float32, Valid=True
distributors: Shape=(10, 12), Data type=torch.float32, Valid=True
disruption_events: Shape=(12, 1), Data type=torch.float32, Valid=True

EDGE TENSOR VALIDATION
('suppliers', 'supplies_to', 'manufacturers'): Shape=(2, 25), Valid indices=True
  Edge features: (25, 8)
('manufacturers', 'ships_via', 'ports'): Shape=(2, 9), Valid indices=True
  Edge features: (9, 4)
('ports', 'route_to', 'ports'): Shape=(2, 50), Valid indices=True
  Edge features: (50, 15)
('ports', 'delivers_to', 'distributors'): Shape=(2, 12), Valid indices=True
  Edge features: (12, 3)
('ports', 'had_disruption', 'disruption_events'): Shape=(2, 12), Valid indices=True

Graph tensor validation completed!


In [35]:
# Save the validated AtmoGraph heterogeneous graph for future GNN development

GRAPH_OUTPUT_DIR = OUTPUT_DIR / "graph"
GRAPH_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

graph_file_path = GRAPH_OUTPUT_DIR / "atmo_graph_heterodata.pt"

torch.save(hetero_data, graph_file_path)

print("Heterogeneous graph saved successfully!")
print("File:", graph_file_path)
print("Node types:", len(hetero_data.node_types))
print("Relationship types:", len(hetero_data.edge_types))
print("Total nodes:", sum(
    hetero_data[node_type].num_nodes
    for node_type in hetero_data.node_types
))
print("Total relationships:", sum(
    hetero_data[edge_type].edge_index.shape[1]
    for edge_type in hetero_data.edge_types
))

Heterogeneous graph saved successfully!
File: D:\AtmoGraph-Supply-Chain-Ripple-Predictor\outputs\processed_features\graph\atmo_graph_heterodata.pt
Node types: 5
Relationship types: 5
Total nodes: 45
Total relationships: 108


In [36]:
# Inspect original disruption and route data to identify prediction targets

datasets_to_inspect = {
    "Disruption Events": nodes["disruption_events"],
    "Ports": nodes["ports"],
    "Port Disruption Relationships": edges["had_disruption"],
    "Shipping Routes": edges["route_to"]
}

for name, df in datasets_to_inspect.items():
    print(f"\n{'=' * 60}")
    print(name)
    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist())
    print("\nSample records:")
    display(df.head(5))


Disruption Events
Shape: (12, 10)
Columns: ['id', 'name', 'year', 'reduction_days', 'shutdown_days', 'recovery_days', 'total_affected_days', 'severity', 'source', 'status']

Sample records:


,id,name,year,reduction_days,shutdown_days,recovery_days,total_affected_days,severity,source,status
0,EVT_001,Typhoon Lekima,2019,3,0,2,5,2,real (Port Disruption DB),historical
1,EVT_002,Hurricane Irene,2011,2,1,3,6,3,real (Port Disruption DB),historical
2,EVT_003,Hurricane Sandy,2012,1,5,3,9,3,real (Port Disruption DB),historical
3,EVT_004,Hurricane Matthew,2016,2,0,2,4,1,real (Port Disruption DB),historical
4,EVT_005,Hurricane Maria,2017,0,0,0,0,0,real (Port Disruption DB),historical



Ports
Shape: (13, 12)
Columns: ['id', 'name', 'unlocode', 'country', 'country_iso3', 'lat', 'lon', 'hist_disruption_events', 'hist_avg_affected_days', 'hist_max_severity', 'geo_source', 'source']

Sample records:


,id,name,unlocode,country,country_iso3,lat,lon,hist_disruption_events,hist_avg_affected_days,hist_max_severity,geo_source,source
0,PORT_INNSA,Nhava Sheva (JNPT),INNSA,India,IND,18.95,72.95,0,NaN,NaN,approximate_manual,reference
1,PORT_INMUN,Mundra,INMUN,India,IND,22.74,69.70,0,NaN,NaN,approximate_manual,reference
2,PORT_INMAA,Chennai,INMAA,India,IND,13.09,80.29,0,NaN,NaN,approximate_manual,reference
3,PORT_INCCU,Kolkata,INCCU,India,IND,22.54,88.33,0,NaN,NaN,approximate_manual,reference
4,PORT_CNSHA,Shanghai,CNSHA,China,CHN,31.23,121.47,1,5.0,2.0,approximate_manual,derived



Port Disruption Relationships
Shape: (12, 3)
Columns: ['start_id', 'end_id', 'source']

Sample records:


,start_id,end_id,source
0,PORT_CNSHA,EVT_001,real (Port Disruption DB)
1,PORT_USNYC,EVT_002,real (Port Disruption DB)
2,PORT_USNYC,EVT_003,real (Port Disruption DB)
3,PORT_USNYC,EVT_004,real (Port Disruption DB)
4,PORT_USNYC,EVT_005,real (Port Disruption DB)



Shipping Routes
Shape: (50, 15)
Columns: ['start_id', 'end_id', 'route_id', 'lead_time_days', 'volume', 'transport_mode', 'distance_km', 'cargo_type', 'avg_delay_days', 'avg_freight_cost_usd', 'disruption_rate', 'delayed_rate', 'avg_port_congestion', 'source', 'dependency_weight']

Sample records:


,start_id,end_id,route_id,lead_time_days,volume,transport_mode,distance_km,cargo_type,avg_delay_days,avg_freight_cost_usd,disruption_rate,delayed_rate,avg_port_congestion,source,dependency_weight
0,PORT_INNSA,PORT_USNYC,R00001,19,8850.984,Rail,8746,Consumer Goods,6.444,4635.075,0.029,0.666,49.708,derived (country route mapped to gateway ports...,0.1855
1,PORT_JPTYO,PORT_CNSHA,R00002,29,22811.299,Air,5420,Agriculture,6.338,3277.016,0.032,0.642,49.222,derived (country route mapped to gateway ports...,0.1775
2,PORT_AUBNE,PORT_CNSHA,R00003,16,6475.579,Rail,12247,Consumer Goods,6.523,6063.867,0.029,0.657,48.437,derived (country route mapped to gateway ports...,0.0658
3,PORT_AUBNE,PORT_CAVAN,R00004,32,9746.301,Road,12143,Energy,6.438,6019.780,0.046,0.623,51.326,derived (country route mapped to gateway ports...,0.0990
4,PORT_GBFXT,PORT_USNYC,R00005,37,14567.679,Sea,14048,Energy,6.413,6795.879,0.030,0.655,48.589,derived (country route mapped to gateway ports...,0.1187
